## Using OpenAI api and Ollama local model llama 3.2

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [17]:
# imports
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI
import ollama

In [2]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [5]:
# set up environment

load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key?")
    
MODEL = MODEL_GPT
openai = OpenAI()


API key looks good so far


In [ ]:

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [11]:
system_prompt = "You are a helpful technical tutor who answers questions about python code, software engineering, data science and LLMs"
user_prompt = "Please give a detailed explanation to the following question: " + question

In [18]:
# gpt-4o-mini answer, with streaming

messages = [
    {'role':'system', 'content':system_prompt},
    {'role':'user', 'content':user_prompt + question}
]


In [13]:
stream = openai.chat.completions.create(
    model=MODEL_GPT,
    messages=messages,
    stream=True
)

response = ""

display_handle = display(Markdown(""), display_id=True)

for chunk in stream:
    response += chunk.choices[0].delta.content or ''
    response = response.replace("'''","").replace("markdown", "")
    update_display(Markdown(response), display_id=display_handle.display_id)

Certainly! Let's break down the code snippet you provided, step by step. The line of code in question is:

```python
yield from {book.get("author") for book in books if book.get("author")}
```

### Components of the Code

1. **Set comprehension:**
   ```python
   {book.get("author") for book in books if book.get("author")}
   ```
   This part of the code is a set comprehension. A set comprehension is similar to list comprehension but creates a set (which is an unordered collection of unique elements) instead of a list.

   - `book.get("author")`: This retrieves the value associated with the key `"author"` from each `book` dictionary in the iterable `books`. If the key does not exist, it returns `None`.

   - `for book in books`: This is iterating over each `book` in the `books` collection.

   - `if book.get("author")`: This acts as a filter, ensuring that only authors that are found (i.e., not `None`) are included in the set.

   The resulting set will contain all unique authors from the `books` collection where an author is present. If the same author appears in multiple books, they will only appear once in the set.

2. **yield from:**
   ```python
   yield from { ... }
   ```
   The `yield from` expression is used within a generator function. It allows the generator to yield all values from an iterable (in this case, the set generated above) one by one, effectively delegating the yielding of values to the iterable.

### What the Code Does

Putting it all together, this line of code does the following:

1. It creates a set of unique authors by:
   - Iterating through each book in the `books` collection.
   - Collecting the authors (by retrieving the `"author"` key).
   - Ignoring any books that do not have an associated author.

2. It yields each author from the created set one by one. This means that when this line is executed in a generator function, it will provide each unique author in the `books` collection as an output.

### Why This Code Is Used

- **Uniqueness:** By using a set comprehension, the code ensures that only unique authors are included, avoiding duplicates that would occur if a list comprehension were used instead.

- **Efficiency in Processing:** By utilizing `yield from`, the code allows for the authors to be generated lazily. This means that rather than generating and returning all authors at once (which could consume a lot of memory), the function will yield one author at a time. This can be particularly useful if the `books` collection is large, as it allows processing to begin immediately and helps keep memory usage low.

### Example

Here's a quick illustrative example. Suppose `books` looks like this:

```python
books = [
    {"title": "Book 1", "author": "Author A"},
    {"title": "Book 2", "author": "Author B"},
    {"title": "Book 3", "author": "Author A"},  # Duplicate author
    {"title": "Book 4"},                         # No author key
    {"title": "Book 5", "author": "Author C"}
]
```

When the comprehension is evaluated, it results in the set `{"Author A", "Author B", "Author C"}`. The `yield from` then yields each author when called in the context of a generator.

In summary, this line of code effectively extracts and yields unique authors from a collection of book dictionaries, considering only those books that actually have an author listed.

In [ ]:
# Llama 3.2 answer

response = ollama.chat(model=MODEL_LLAMA, messages=messages)
reply = response['message']['content']
display(Markdown(reply))

I'd be happy to explain this code.

**Explanation of the Code**

The code snippet you provided is using Python's generator expression to extract the author's name from a list of books, while filtering out books without an author.

Here's a breakdown of the code:

* `yield from`: This is a Python 3.3+ syntax that allows you to delegate to a sub-generator. It allows you to use the `yield` keyword to produce a series of values from another iterable.
* `{...}`: This is a dictionary comprehension, which creates a new dictionary containing the results of the expression inside the curly brackets.
* `book.get("author")`: This is a method call on the `book` dictionary, which returns the value associated with the key `"author"`. If the key doesn't exist, it returns `None`.
* `for book in books if book.get("author")`: This is a filter clause, which only includes books that have an author. The `if` clause is a predicate function that takes a book and returns `True` if the book has an author, and `False` otherwise.

**Why the Code is Written this Way**

This code is written in this way because it's more memory-efficient than creating a new list or dictionary with all the author's names. By using a generator expression and the `yield from` syntax, we can produce a sequence of values without having to store them all in memory at once.

Here's an example to illustrate this:

Suppose we have a list of books, where each book is a dictionary with an `"author"` key:
```python
books = [
    {"title": "Book 1", "author": "John Doe"},
    {"title": "Book 2", "author": "Jane Smith"},
    {"title": "Book 3", "author": None}
]
```
If we use a regular list comprehension to extract the author's names, we'll create a new list with all the author's names, even if some books don't have an author:
```python
authors = [book["author"] for book in books]
```
This will create a list with all the author's names, even if some books don't have an author:
```python
authors = ["John Doe", "Jane Smith", None]
```
By using the `yield from` syntax, we can produce a generator that only yields the author's names for books that have an author:
```python
authors = yield from (book["author"] for book in books if book.get("author"))
```
This will produce a generator that yields only the author's names for books that have an author:
```python
authors = yield from (
    "John Doe"
    "Jane Smith"
)
```
This is more memory-efficient because we don't have to store all the author's names in memory at once. Instead, we can generate them on the fly as we need them.